<!--- sf-header --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fscale-forecasting%2Fmain%2Fnotebooks%2F11_auto_forecaster_and_autofocus.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
</tr>
</table>
<br clear="left"/>

> **Run in Colab Enterprise:** click the badge to import this notebook, pick a runtime, and
> **Run all**. Runs 100% locally in-process (`where="local"`, zero GCP billing required) on the **`sf-main`** runtime template (Python 3.11) or any local `.venv`. See
> [`docs/notebook_runtimes.md`](https://github.com/statmike/scale-forecasting/blob/main/docs/notebook_runtimes.md)
> for the per-notebook template mapping and the headless acceptance harness.

# 11 · Glass-Box Auto-Forecaster, Exposure Tiers, Multi-Branch `autofocus` & Evolutionary Search (`evolve`)

When you want to forecast a new dataset without spending days hand-tuning configurations—or handing your data to an opaque black-box AutoML tool—**`Forecaster.build_config()`**, **`Forecaster.auto()`**, and **`Forecaster.evolve()`** give you a **100% transparent ("glass-box") automated forecasting workbench**:

1. **Every choice is explained in plain English:** You get a Step 0 Data Profile (`Forecaster.profile_data()` / `f.profile_df()`), a row-by-row compiler rationale (`bp.rationale_df`), and a post-run **7-Question Practitioner Assessment (`f.practitioner_assessment_df()`)** answering the recurring questions forecasters ask in real-world reviews (*Is this model better than Model X? Do ensembles help? Do covariates help or overfit? Does `@global` beat `@local`? Is reconciliation done properly? Does accuracy hold across the horizon? How do I trust the forecast?*).
2. **Evolved with AlphaEvolve on Gemini Enterprise (Zero LLM Calls at Runtime):** The tier defaults and `autofocus` rules in `scale_forecasting.auto_policy` were discovered using **AlphaEvolve on Gemini Enterprise** (`gemini-3.8-flash` + `gemini-4-argon`) across 20 ground-truth test panels and compiled into pure, deterministic Python—so runtime execution makes **zero LLM calls** and is 100% reproducible.
3. **Opt-In Runtime Evolutionary Search & Post-Run Super-Ensembling (`Forecaster.evolve()`):** When you want to push accuracy even further—either on a single completed run (`f.evolve(target="ensemble")`), across a pool of prior runs (`Forecaster.evolve(from_runs=[...], target="ensemble")`), or by evolving custom domain features/models (`target="features" | "model"`)—the opt-in `scale_forecasting.evolve` harness mutates bounded `# EVOLVE-BLOCK` code under a **3-Gate Anti-Overfitting Ratchet** and **`+1e6` Future-Leakage Sentinel**.
4. **What is a "DGP" (Data-Generating Process)?** Throughout this notebook you will see the term **DGP (Data-Generating Process)**. That simply means a **synthetic test dataset built from a known mathematical recipe** (where we planted known seasonal cycles, stockouts, price spikes, hierarchies, or fake random-noise columns ahead of time so we can prove `autofocus` and `evolve` find the real signal and drop the fake noise).

### At a glance
| Dimension | Details |
| :--- | :--- |
| **Learning outcomes** | Run a Step 0 Data Profile (`Forecaster.profile_data()` / `inspect_panel_for_auto()`), generate transparent `ConfigBlueprint`s across all 4 exposure tiers (`first_light`, `wide_field`, `deep_sky`, `deep_field`) and 5 `n_series` scale regimes (`micro` to `hyperscale`), pin your own model/metric/baseline constraints, preview approximate runtime & cost estimates across all 7 cloud runtimes, run multi-branch `autofocus=True` tournaments that borrow finished models across passes, answer the **7 Practitioner Trust Questions** (`f.practitioner_assessment_df()`, `f.compare_models()`, `f.baseline_value_df()`, `f.hierarchy_value_df()`), step through passes interactively (`step_by_step=True` + `f.step()`), and run **post-run / multi-run evolutionary super-ensembling & causal feature verification (`Forecaster.evolve()`)**. |
| **Prerequisites** | None — runs 100% offline (`where="local"`, `backend="mock"`) with `scale-forecasting[notebook,models-stats,models-trees]`; Act 6 also shows live cloud `backend="alphaevolve"` and `backend="vertex_gemini"` (`gemini-3.8-flash` / `gemini-4-argon`) usage. |
| **Estimated time** | ~50–65 seconds end-to-end. |
| **Estimated run cost** | **`\$0.00` USD** (runs locally in memory; Act 3 also previews approximate cloud cost ballparks across Spark, Ray, Vertex AI, GCE, GKE, AutoML, and BigQuery). |
| **Ongoing cost** | **Zero** — no cloud resources are provisioned. |

```mermaid
flowchart TD
    subgraph Step0["Step 0: Data Profile & Readiness Scan"]
        S0["Forecaster.profile_data() / inspect_panel_for_auto()<br/>7 dimensions: scale_regime · SBC demand · seasonality & log1p<br/>breaks & spikes · hygiene & calendar · covariates · hierarchy"]
    end
    subgraph Dials["1. Your Two Dials + Optional Pins"]
        D1["Where to Run (where): local · bigquery · cloud_cpu · cloud_all"]
        D2["Exposure Tier (tier): first_light · wide_field · deep_sky · deep_field"]
    end
    subgraph Compiler["2. Glass-Box Compiler"]
        C2["Forecaster.build_config()<br/>Editable RunConfig + Plain-English Rationale + 4 Baseline Anchors"]
    end
    subgraph Loop["3. Multi-Branch autofocus=True + 7-Question Assessment"]
        L1["Pass 1 Baseline"]
        L2["Parallel What-If Branches (2A / 2B / 2C)<br/>Log transform · Drop fake covariates · Specialists"]
        L3["Pass 3 Synthesis, Champion Ratchet & Practitioner Assessment<br/>Borrows Pass 1 models + blends winners + f.practitioner_assessment_df()"]
        L1 --> L2 --> L3
    end
    subgraph Evolve["4. Optional: Evolutionary Search & Super-Ensembling (Forecaster.evolve)"]
        E1["Post-Run (f.evolve) · Multi-Run Pool (from_runs / from_run_ids) · Custom # EVOLVE-BLOCK<br/>3-Gate Anti-Overfitting Ratchet + +1e6 Future-Leakage Sentinel"]
    end
    Step0 --> Dials --> Compiler --> Loop --> Evolve
```

## Act 1 · Setup & the 20 Synthetic "Known-Recipe" Test Datasets (DGPs)

In real-world data, you never know for sure whether an extra promotion column is true signal or random coincidence. To test our auto-forecaster rigorously, `scale_forecasting.benchmarks` includes **20 synthetic Data-Generating Process (DGP) benchmark datasets**—each built from a known mathematical recipe (such as exponential growth, zero-heavy spare-parts demand, 2 real drivers mixed with 3 fake noise columns, 4 all-decoy noise columns where a univariate model should win, multi-level hierarchies, capped stockout days, hourly multi-seasonal load, or monthly regime drift) so we can verify that `autofocus` makes the right call on every regime.

In [1]:
# Cloud bootstrap: clone + install the LOCKED dependency set (uv.lock) so
# `import scale_forecasting` resolves against the exact versions every other surface runs.
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = os.environ.get("SF_REPO_URL", "https://github.com/statmike/scale-forecasting.git")
REPO_DIR = os.environ.get("SF_REPO_DIR", "scale-forecasting")
EXTRAS = ['notebook', 'models-stats', 'models-trees']

if importlib.util.find_spec("scale_forecasting") is None:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    uv = shutil.which("uv")
    if uv is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        uv = shutil.which("uv") or "uv"
    reqs = os.path.abspath(os.path.join(REPO_DIR, "colab-requirements.txt"))
    extra_flags = [f for e in EXTRAS for f in ("--extra", e)]
    subprocess.run(
        [uv, "export", "--frozen", "--no-emit-project", "--no-hashes", "--no-dev", *extra_flags, "-o", reqs],
        cwd=REPO_DIR,
        check=True,
    )
    target = os.path.abspath(os.path.join(REPO_DIR, ".colab-deps"))
    subprocess.run(
        [uv, "pip", "install", "--python", sys.executable, "--target", target, "-r", reqs],
        check=True,
    )
    sys.path.insert(0, target)
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))
    print("installed scale_forecasting from", REPO_DIR)
else:
    print("scale_forecasting already importable — skipping bootstrap")

import pandas as pd
from scale_forecasting import (
    EXPOSURE_TIERS,
    Forecaster,
    RunConfig,
    benchmark_catalog,
    calibrate_duration_from_frames,
    inspect_panel_for_auto,
    load_benchmark,
    predict_run_duration,
)

cat_df = benchmark_catalog()
cat_df

scale_forecasting already importable — skipping bootstrap


,name,title,freq,horizon,n_series,history_length,expected_regime,true_transform,true_covariates,decoy_covariates,has_horizon_crossover,has_structural_break,censored_col,hierarchy_id_cols,already_optimal_at_pass_1
0,dgp_01_smooth_seasonal,Smooth Additive Trend + Weekly Seasonal Harmonics,D,14,12,140,smooth,none,[],[],False,False,None,[],False
1,dgp_02_multiplicative_log,Exponential Growth with Multiplicative Varianc...,D,14,12,140,smooth,log1p,[],[],False,False,None,[],False
2,dgp_03_intermittent_lumpy,Sparse Intermittent & Lumpy Integer Spare-Part...,D,14,12,140,intermittent,none,[],[],False,False,None,[],False
3,dgp_04_true_vs_decoy_covs,2 True Exogenous Drivers + 3 Pure-Noise Decoy ...,D,14,12,140,smooth,none,"[promo_depth, price_index]","[noise_a, noise_b, noise_c]",False,False,None,[],False
4,dgp_05_horizon_crossover,Short-Memory AR(1) Shock + Long-Horizon Season...,D,14,12,140,smooth,none,[promo_flag],[],True,False,None,[],False
5,dgp_06_structural_break_spikes,Permanent Historical Level Shift + Isolated 5-...,D,14,12,140,erratic,none,[],[],False,True,None,[],False
6,dgp_07_censored_stockouts,Right-Censored Demand Under Inventory Stockouts,D,14,12,140,smooth,none,[promo_flag],[],False,False,stockout_flag,[],False
7,dgp_08_hierarchical_nonneg,Two-Level Region -> Store Hierarchy with Non-N...,D,14,12,140,smooth,none,[],[],False,False,None,"[region, store]",False
8,dgp_09_cross_series_global,Shared Cross-Series Promo & Price Elasticity C...,D,14,16,112,smooth,none,"[promo_depth, price_index]",[],False,False,None,[],False
9,dgp_10_newsvendor_service,Asymmetric Newsvendor Replenishment (9:1 Stock...,D,14,12,140,smooth,none,[],[],False,False,None,[],False


## Act 2 · Glass-Box Config Generation Across the 4 Exposure Tiers & Pinning Your Own Rules (`Forecaster.build_config`)

Just like a telescope starts with a quick **"first light"** calibration snapshot before taking a **"wide field"** or **"deep field"** exposure, `Forecaster.build_config()` lets you choose how deeply to search via `tier`.

It profiles your dataset once (`inspect_panel_for_auto`) and returns a validated `RunConfig` paired with a `rationale_df` table explaining every choice in plain English. You can also **pin any rule your team cares about** (`models`, `include_models`, `exclude_models`, `decision_metric`, `n_folds`, `hpo`, `ensemble`, `explainability`, `calibration`, `overrides`, or `blueprint.with_overrides(**kwargs)`), and every pinned setting is preserved and marked `[PINNED]` in the rationale table.

In [2]:
ds_cov = load_benchmark("dgp_04_true_vs_decoy_covs", n_series=4)
profile = inspect_panel_for_auto(ds_cov.panel_df)

print("=== Step 0 Data Profile & Readiness Scan (dgp_04_true_vs_decoy_covs) ===")
display(profile.to_df())

tier_rows = []
for tier_name in EXPOSURE_TIERS:
    bp = Forecaster.build_config(
        profile,
        tier=tier_name,
        where="cloud_all",
        goal="accuracy",
        only_available_models=False,
    )
    tier_rows.append(
        {
            "tier": tier_name,
            "n_models": len(bp.config.models),
            "models": ", ".join(bp.config.models),
            "n_folds": bp.config.backtest.n_folds,
            "hpo_enabled": bp.config.hpo.enabled,
            "ensemble_strategies": ", ".join(bp.config.ensemble.strategies),
            "pred_p50_min": round(bp.estimated_minutes_p50, 2),
            "est_usd": round(bp.cost_estimate_usd, 3),
        }
    )

print("\n=== Exposure Tier Comparison (dgp_04_true_vs_decoy_covs) ===")
display(pd.DataFrame(tier_rows))

wide_bp = Forecaster.build_config(
    ds_cov.panel_df,
    tier="wide_field",
    where="local",
    goal="accuracy",
    include_models=["regression_lags"],
    exclude_models=["xgboost"],
    decision_metric="wape",
    n_folds=2,
)
print("\n" + wide_bp.summary())
display(wide_bp.rationale_df)

=== Step 0 Data Profile & Readiness Scan (dgp_04_true_vs_decoy_covs) ===


,dimension,signal,measured_value,implication
0,scale_and_shape,n_series · history · scale_regime · cross_seri...,4 series (micro) | freq='D' | median_T=140 (mi...,"Per-series local statistical, tree, and zero-s..."
1,demand_regime,dominant_regime · zero_fraction · ADI · CV² · ...,regime='smooth' | zeros=0.0% | ADI=1.00 | CV²=...,Smooth/continuous series support WAPE scoring ...
2,seasonality_and_scaling,seasonal_acf · level_vol_corr · end_to_start_r...,acf_season=0.52 | level_vol_corr=0.39 | growth...,Additive variance regime; standard linear targ...
3,breaks_and_outliers,structural_break_share · outlier_spike_share,breaks=0.0% | spikes=0.0%,Clean historical trajectory; no aggressive out...
4,hygiene_and_calendar,leading_zeros · missing_gaps · censored_col,leading_zeros=0.0% | date_gaps=0.0% | censored...,Contiguous date grid with no pre-launch zero p...
5,covariates,future · past · static covariates,"5 column(s): promo_depth, price_index, noise_a...",Benchmarks covariate models (@cov_all) against...
6,hierarchy_topology,hierarchy_id_cols · node_count · non_negative,Flat panel (no hierarchy columns) | non_negati...,Independent series; enforces output.floor=0.0 ...



=== Exposure Tier Comparison (dgp_04_true_vs_decoy_covs) ===


,tier,n_models,models,n_folds,hpo_enabled,ensemble_strategies,pred_p50_min,est_usd
0,first_light,4,"naive_seasonal, theta, lightgbm, regression_lags",1,False,"inverse_error, mean",3.94,0.054
1,wide_field,11,"naive_seasonal, theta, holtwinters, fft, light...",2,False,"inverse_error, nnls, ridge, champion",4.15,0.264
2,deep_sky,17,"naive_seasonal, theta, holtwinters, fft, autoe...",3,True,"inverse_error, nnls, ridge, champion",4.18,0.295
3,deep_field,23,"naive_seasonal, theta, holtwinters, fft, autoe...",3,True,"inverse_error, nnls, ridge, champion",4.85,0.325



Glass-Box Blueprint : auto-wide-field (auto-wide-field-06a71bb70738)
Exposure Tier       : wide_field (Wide Field (~2m local / ~2m BQ / ~7-16m cloud, Default))
Sandbox Envelope    : where='local' | goal='accuracy' | scale='micro'
Panel Profile       : 4 series | freq='D' | median_T=140 | H=14 | regime='smooth'
Baseline Ladder     : naive_seasonal (naive_anchor), theta (plain_univariate_stat), chronos_bolt (zero_shot_foundation), lightgbm@univariate (same_model_univariate)
Selected Models (9) : naive_seasonal, theta, holtwinters, fft, lightgbm, regression_lags, chronos_bolt, lightgbm@univariate, regression_lags@univariate
Predicted Duration  : p50=1.6s (0.03m) | p90=2.7s (0.04m) | bottleneck=ml | est_cost=$0.00


,category,parameter,value,evidence,rationale
0,scale_regime,scale_regime,micro (Micro Panel (1–14 series)),"n_series=4, n_rows=560","Per-series local statistical, tree, and zero-s..."
1,models,naive_seasonal,include (statistical),"freq=D, median_acf=0.52",Establishes the zero-skill seasonal benchmark ...
2,models,theta,include (statistical),median_n_obs=140,Robust trend + seasonal statistical baseline.
3,models,holtwinters,include (statistical),median_acf=0.52,Captures damped trend and seasonal harmonics c...
4,models,fft,include (statistical),dominant_regime=smooth,Fast Fourier harmonic extrapolation for period...
5,models,lightgbm,include (ml),"n_series=4, fut_covs=5",GBDT learner capturing nonlinear lag and exoge...
6,models,regression_lags,include (ml),end_to_start_ratio=1.12,Fast ridge lag regression providing linear ext...
7,models,chronos_bolt,include (deep_learning),"has_cold_start=False, scale_regime=micro",Zero-shot pretrained foundation baseline excel...
8,output,output.floor,0.0,min(y) >= 0.0 across 100% of historical observ...,Enforces physical non-negativity on point fore...
9,backtest,backtest.decision_metric,wape,"goal='accuracy', intermittent=False",Optimizes model selection and ensemble weights...


## Act 3 · Approximate Runtime & Cloud Cost Estimator Across All 7 Runtimes (`predict_run_duration`)

Because `scale-forecasting` launches **one independent job per active model family in parallel** (`statistical`, `ml`, `deep_learning`, `automl`, `native`) and then runs a quick ensemble step, total wall-clock time is governed by the **slowest active family**, not the sum of all families:

$$\text{Total Time} \approx \max_{\text{family}} \big( \text{Startup / Provisioning} + \text{Setup Overhead} + \text{Model Fitting Time} \big) + \text{Ensemble Step}$$

> **Note — Approximate Planning Estimates Only:** All durations and USD figures below are **approximate order-of-magnitude planning estimates** calibrated from 606 historical jobs (`5.59M` fitted cells) in `us-central1`—not billing quotes. Actual runtimes and cloud costs vary by region, quota availability, data shape, BigQuery pricing edition, and over time, and do not capture every ancillary storage, query, or networking charge.

Below we compare approximate P50/P90 wall-clock duration and estimated compute cost across all 7 cloud runtimes plus local in-process execution for a 500-series workload.

In [3]:
runtime_configs = [
    ("local (in-process)", ["theta", "lightgbm"], {}, "local"),
    ("bigquery (ARIMA_PLUS + TimesFM)", ["arima_plus", "timesfm"], {}, None),
    (
        "spark (Serverless)",
        ["theta", "holtwinters", "lightgbm"],
        {"families": {"statistical": {"runtime": "spark"}, "ml": {"runtime": "spark"}}},
        None,
    ),
    (
        "ray (Vertex AI Ray)",
        ["theta", "lightgbm", "xgboost"],
        {
            "families": {
                "statistical": {"runtime": "ray", "ray_mode": "vertex"},
                "ml": {"runtime": "ray", "ray_mode": "vertex"},
            }
        },
        None,
    ),
    (
        "vertex (CustomJob L4 GPU)",
        ["tide@global", "tsmixer@global"],
        {"families": {"deep_learning": {"runtime": "vertex", "hardware": "gpu", "gpu_type": "L4"}}},
        None,
    ),
    (
        "gce (Single-VM CPU)",
        ["theta", "lightgbm"],
        {
            "families": {
                "statistical": {"runtime": "gce", "workers": 1},
                "ml": {"runtime": "gce", "workers": 1},
            }
        },
        None,
    ),
    (
        "gke (Indexed Job)",
        ["theta", "lightgbm"],
        {
            "families": {
                "statistical": {"runtime": "gke", "gke_mode": "job", "workers": 2},
                "ml": {"runtime": "gke", "gke_mode": "job", "workers": 2},
            }
        },
        None,
    ),
    (
        "vertex_automl (Training Job)",
        ["vertex_tide"],
        {"families": {"automl": {"runtime": "vertex_automl", "automl_mode": "training_job"}}},
        None,
    ),
]

dur_rows = []
for label, models_list, comp_dict, where_override in runtime_configs:
    cfg_rt = RunConfig.model_validate(
        {
            "run_name": "dur-compare",
            "data": {"source_table": "source_series", "horizon": 14},
            "models": models_list,
            "compute": comp_dict,
        }
    )
    est = predict_run_duration(cfg_rt, n_series=500, median_n_obs=365.0, where=where_override)
    dur_rows.append(
        {
            "runtime_surface": label,
            "critical_path_family": est.critical_path_family,
            "provision_p50_s": round(est.provision_p50_seconds, 1),
            "execute_p50_s": round(est.execute_p50_seconds, 1),
            "total_p50_min": round(est.total_p50_minutes, 2),
            "total_p90_min": round(est.total_p90_minutes, 2),
            "estimated_usd": round(est.estimated_usd, 3),
        }
    )

pd.DataFrame(dur_rows)

,runtime_surface,critical_path_family,provision_p50_s,execute_p50_s,total_p50_min,total_p90_min,estimated_usd
0,local (in-process),ml,0.0,49.3,0.60,1.08,0.000
1,bigquery (ARIMA_PLUS + TimesFM),native,13.1,39.8,0.95,2.45,0.009
2,spark (Serverless),ml,64.4,333.8,3.92,5.01,0.439
3,ray (Vertex AI Ray),ml,221.3,172.7,5.21,15.65,1.090
4,vertex (CustomJob L4 GPU),deep_learning,196.4,78.1,4.64,6.54,0.097
5,gce (Single-VM CPU),ml,239.5,114.6,5.10,5.91,0.057
6,gke (Indexed Job),ml,79.8,88.3,2.18,8.51,0.051
7,vertex_automl (Training Job),automl,1930.0,1807.2,62.35,113.62,227.313


## Act 4 · Multi-Branch `autofocus=True` Tournament & Champion Ratchet Across 5 Test Recipes (`Forecaster.auto`)

Now let's run `Forecaster.auto(..., tier="first_light", autofocus=True, autofocus_mode="dag", max_passes=3)` across five synthetic test recipes (DGPs) to watch how `autofocus` diagnoses Pass 1 errors, tests improvements in parallel branches, **borrows already-fitted models from Pass 1**, and locks in the best pass with its **Champion Ratchet**:

1. **`dgp_02_multiplicative_log` (Swings grow with volume):** Pass 1 fits linear models, spots that errors grow with series level, and **Pass 2 adds `@transform_log1p` log-scale models while borrowing Pass 1's linear models**, cutting WAPE by `>20%`.
2. **`dgp_03_intermittent_lumpy` (Zero-heavy spare-parts demand):** Spots that most days have zero demand and adds intermittent specialists (`croston@classic`, `croston@sba`, `naive_mean`, `regime_routing=True`) under `decision_metric="mase"`.
3. **`dgp_04_true_vs_decoy_covs` (Multi-Branch DAG `1 -> 2A -> 2B -> 3`):** Runs parallel Level-2 branches—Branch `2A` (scale/clean-up) and Branch `2B` (which spots and drops the 3 fake noise columns `[noise_a, noise_b, noise_c]`)—and merges the winners in Pass `3`, cutting WAPE by `>45%`.
4. **`dgp_08_hierarchical_nonneg` (Store -> Region hierarchy):** Runs parallel branches and reconciles both base and ensemble forecasts in Pass `3` via `mint_nonneg` so store forecasts add up to the exact regional total (`verify_coherence == 0.0`).
5. **`dgp_12_already_optimal_white_noise` (Pure random noise around a flat mean):** Sees that Pass 1 residuals are already random white noise, drops continuation confidence to `0.12` (`< 0.35`), and **stops immediately after Pass 1 with zero wasted compute**.

In [4]:
summary_rows = []
sessions = {}

for b_name in [
    "dgp_02_multiplicative_log",
    "dgp_03_intermittent_lumpy",
    "dgp_04_true_vs_decoy_covs",
    "dgp_08_hierarchical_nonneg",
    "dgp_12_already_optimal_white_noise",
]:
    ds = load_benchmark(b_name, n_series=4)
    f = Forecaster.auto(
        ds.panel_df,
        tier="first_light",
        where="local",
        autofocus=True,
        autofocus_mode="dag",
        max_passes=3,
    )
    sessions[b_name] = f
    passes = f.auto_session.passes
    p1 = passes[0]
    champ = f.auto_session.champion_record()
    rel_improvement = (p1.best_score - champ.best_score) / max(1e-6, p1.best_score) * 100.0
    summary_rows.append(
        {
            "benchmark": b_name,
            "dag_passes": " -> ".join(p.pass_id for p in passes),
            "pass_1_champion": f"{p1.best_model} ({p1.best_score:.4f})",
            "session_champion": f"[{champ.pass_id}] {champ.best_model} ({champ.best_score:.4f})",
            "relative_gain_pct": round(rel_improvement, 2),
            "fva_vs_naive_pct": round(champ.fva_vs_naive_pct, 2),
            "final_stop_reason": f.preview_next_loop().stop_reason,
        }
    )

display(pd.DataFrame(summary_rows))

print("\n=== Multi-Branch DAG Pass Ledger: dgp_04_true_vs_decoy_covs ===")
display(sessions["dgp_04_true_vs_decoy_covs"].autofocus_ledger_df())

print("\n=== Structured Parameter Diffs Across DAG Branches (`autofocus_diff_df`) ===")
display(sessions["dgp_04_true_vs_decoy_covs"].autofocus_diff_df())
print(
    "Retained future_covariates in Champion config:",
    sessions["dgp_04_true_vs_decoy_covs"].config.features.future_covariates,
)

,benchmark,dag_passes,pass_1_champion,session_champion,relative_gain_pct,fva_vs_naive_pct,final_stop_reason
0,dgp_02_multiplicative_log,1 -> 2,regression_lags (0.1368),[2] ensemble_champion (0.1208),11.73,44.00,Residuals are statistically clean (white-noise...
1,dgp_03_intermittent_lumpy,1 -> 2 -> 3,lightgbm (0.9861),[3] ensemble_champion (0.9263),6.06,31.49,Reached max_passes=3 for the selected exposure...
2,dgp_04_true_vs_decoy_covs,1 -> 2A -> 2B -> 3,ensemble_inverse_error (0.0498),[3] ensemble_champion (0.0398),20.16,63.08,Reached max_passes=3 for the selected exposure...
3,dgp_08_hierarchical_nonneg,1 -> 2,theta (0.0496),[2] ensemble_champion (0.0489),1.31,25.79,Residuals are statistically clean (white-noise...
4,dgp_12_already_optimal_white_noise,1,theta (0.0180),[1] theta (0.0180),0.00,16.52,Residuals are statistically clean (white-noise...



=== Multi-Branch DAG Pass Ledger: dgp_04_true_vs_decoy_covs ===


,pass_index,pass_id,parent_pass_ids,branch_hypothesis,run_id,tier,models_total,models_borrowed,models_executed,champion_model,...,champion_is_univariate,beaten_challengers,dm_pvalue,predicted_p50_s,actual_wall_s,continue_confidence,expected_next_gain_pct,mutations_applied,is_session_champion,stop_or_continue_reason
0,1,1,root,Initial Tier Exposure,auto-first-light-f350285c8399,first_light,4,0,4,ensemble_inverse_error,...,False,(none),0.0,0.67,3.31,0.920,17.00,initial_tier_compilation,False,"Pass 2 approved (confidence=0.92 >= 0.35, expe..."
1,2,2A,1,Target Transform & Structural Hygiene,auto-first-light-p2a-7a2dd575f8bc,first_light,7,4,3,ensemble_champion,...,False,(none),0.0,1.21,9.27,0.602,8.61,inject_transform_log1p,False,"Pass 3 approved (confidence=0.60 >= 0.35, expe..."
2,2,2B,1,Covariate Pruning & Direct Horizon Specialists,auto-first-light-p2b-ba0c84b2d73e,first_light,4,2,2,ensemble_champion,...,False,(none),0.0,0.79,5.46,0.644,7.28,prune_decoy_covariates,False,"Pass 3 approved (confidence=0.64 >= 0.35, expe..."
3,3,3,"2A, 2B",Level-3 Tournament Synthesis & Learned Stacking,auto-first-light-p3-synth-b0905e04d127,first_light,7,5,2,ensemble_champion,...,False,(none),0.0,1.17,5.07,0.000,0.00,synthesize_winning_dag_branches,True,Reached max_passes=3 for the selected exposure...



=== Structured Parameter Diffs Across DAG Branches (`autofocus_diff_df`) ===


,pass_index,pass_id,parent_pass_ids,branch_hypothesis,action,parameter,old_value,new_value,champion_model,champion_score,delta_vs_pass1_pct,is_session_champion,evidence,rationale
0,1,1,root,Initial Tier Exposure,initial_tier_compilation,models,(none),"naive_seasonal, theta, lightgbm, regression_lags",ensemble_inverse_error,0.0498,0.00,False,tier='first_light',Initial Glass-Box exposure tier model & featur...
1,2,2A,1,Target Transform & Structural Hygiene,inject_transform_log1p,models,linear_scale_only,"lightgbm@transform_log1p, regression_lags@tran...",ensemble_champion,0.0411,17.61,False,"residual_heteroskedasticity=0.66, level_vol_co...",Residuals exhibit level-dependent variance; in...
2,2,2B,1,Covariate Pruning & Direct Horizon Specialists,prune_decoy_covariates,features.future_covariates,"['promo_depth', 'price_index', 'noise_a', 'noi...","['promo_depth', 'price_index']",ensemble_champion,0.0416,16.42,False,"noisy_covariates=['noise_a', 'noise_b', 'noise...",Pruned 3 zero/negative-value decoy covariate(s...
3,3,3,"2A, 2B",Level-3 Tournament Synthesis & Learned Stacking,synthesize_winning_dag_branches,models + features + ensemble.strategies,"branches=['2A', '2B']","merged=['2A', '2B'], strategies=['inverse_erro...",ensemble_champion,0.0398,20.16,True,"winning_branches=['2A', '2B'], actions=['injec...",Synthesizes all non-degrading Level-2 hypothes...


Retained future_covariates in Champion config: ['promo_depth', 'price_index']


## Act 5 · Step-by-Step Human-in-the-Loop, 7-Question Practitioner Assessment, Baseline Value Ladder & Hierarchy Audit

Want to review what `autofocus` plans to do before it runs Pass 2? Set `step_by_step=True`.

`Forecaster.auto()` runs Pass 1 and pauses so you can inspect `f.preview_next_loop()`—showing the exact proposed config changes, which Pass 1 models will be borrowed from cache vs. trained fresh, and the approximate duration of the next pass—before calling `f.step()`. At any point:
- **`f.practitioner_assessment_df()`** answers the **7 recurring questions forecasters ask in real-world conversations** (*Is this model better than Model X? Do ensembles help? Do covariates help or overfit? Does `@global` beat `@local`? Is reconciliation done properly? Does accuracy hold across the horizon? How do I trust the forecast?*).
- **`f.compare_models()`** compares the champion (or any `model_a`) head-to-head against every candidate (`model_b`) with FVA %, series win rate %, and Diebold–Mariano $p$-values.
- **`f.baseline_value_df()`** compares your session champion against the 4-anchor baseline ladder (`naive_seasonal`, `theta`, `arima_plus@univariate`, and zero-shot foundation models)—and if a plain univariate model wins (like on `dgp_13_all_decoy_univariate_wins`), it explicitly lists every complex multivariate challenger it defeated.
- **`f.hierarchy_value_df()`** reports base vs. reconciled accuracy at every level of a hierarchical tree (`Root`, `Intermediate`, `Leaf`).
- **`f.explain_auto_decisions_df()`** gives you a single table combining every Pass 1 compilation choice and every `autofocus` follow-up change.

In [5]:
ds_step = load_benchmark("dgp_06_structural_break_spikes", n_series=3)

# 1. Run Pass 1 only in interactive step-by-step mode
f_interactive = Forecaster.auto(
    ds_step.panel_df,
    tier="first_light",
    where="local",
    autofocus=True,
    step_by_step=True,
    max_passes=2,
)
print("After Pass 1:")
display(f_interactive.autofocus_ledger_df())

# 2. Preview what Pass 2 will change and why
preview = f_interactive.preview_next_loop()
print(
    f"\nNext Loop Preview -> should_continue={preview.should_continue} | "
    f"confidence={preview.confidence:.2f} | expected_gain=+{preview.expected_gain_pct:.1f}%"
)
print("Reason:", preview.reason)
display(preview.mutations_df)

# 3. Approve and execute Pass 2 via f.step()
f_interactive.step()
print("\nAfter Approving Pass 2 (`f_interactive.step()`):")
display(f_interactive.autofocus_ledger_df())
display(f_interactive.baseline_value_df())
display(f_interactive.explain_auto_decisions_df())

# 4. Answer the 7 Practitioner Questions & Head-to-Head Model Comparison (on dgp_04)
print("\n7-Question Practitioner Assessment (`dgp_04_true_vs_decoy_covs`):")
display(sessions["dgp_04_true_vs_decoy_covs"].practitioner_assessment_df())
print("\nHead-to-Head Model Comparison (`f.compare_models()` on `dgp_04_true_vs_decoy_covs`):")
display(sessions["dgp_04_true_vs_decoy_covs"].compare_models())

# 5. Inspect hierarchy reconciliation lift on a hierarchical panel (dgp_08)
print("\nHierarchy Reconciliation Value Audit (`dgp_08_hierarchical_nonneg`):")
display(sessions["dgp_08_hierarchical_nonneg"].hierarchy_value_df())

After Pass 1:


,pass_index,pass_id,parent_pass_ids,branch_hypothesis,run_id,tier,models_total,models_borrowed,models_executed,champion_model,...,champion_is_univariate,beaten_challengers,dm_pvalue,predicted_p50_s,actual_wall_s,continue_confidence,expected_next_gain_pct,mutations_applied,is_session_champion,stop_or_continue_reason
0,1,1,root,Initial Tier Exposure,auto-first-light-45e078f441ab,first_light,4,0,4,naive_seasonal,...,True,"lightgbm, regression_lags",1.0,0.48,2.14,0.96,23.1,initial_tier_compilation,True,"Pass 2 approved (confidence=0.96 >= 0.35, expe..."



Next Loop Preview -> should_continue=True | confidence=0.96 | expected_gain=+23.1%
Reason: Pass 2 approved (confidence=0.96 >= 0.35, expected FVA gain ~+23.1% via inject_direct_horizon_models, enable_level_shift, enable_stl_resid_outlier_scrubbing).


,action,parameter,old_value,new_value,evidence,rationale
0,inject_direct_horizon_models,models + ensemble.strategies,"['inverse_error', 'mean']","regression_lags@direct, lightgbm@direct + hori...","horizon_crossover_step=4, late_horizon_degrada...",Recursive lag rollouts degrade beyond short le...
1,enable_level_shift,features.level_shift,False,True,change_point_share=100.0%,Post-fit change-point audit flagged structural...
2,enable_stl_resid_outlier_scrubbing,features.outlier_policy,none,stl_resid,anomaly_share=100.0%,Residual anomaly audit flagged extreme histori...



After Approving Pass 2 (`f_interactive.step()`):


...

,question_id,question,verdict,answer,evidence,recommendation
0,model_vs_baseline,1. Is the winning model genuinely better than ...,verified_lift,Champion 'ensemble_champion' reduced wape by +...,"champion=ensemble_champion (wape=0.0398), fva_...",Inspect f.baseline_value_df() and f.compare_mo...
1,ensemble_vs_single,"2. Do ensembles help here, or is a single mode...",ensemble_helps,Yes — 'ensemble_champion' outperformed the bes...,best_single=regression_lags@transform_log1p (0...,Ensembles must beat the best single model by t...
2,covariates_vs_overfitting,"3. Do covariates help, or are they just overfi...",selective_signal_kept,"True drivers ['promo_depth', 'price_index'] im...","kept=['promo_depth', 'price_index'], pruned_de...",Inspect f.attributions_df() to see per-feature...
3,global_vs_local,4. Does cross-series @global training beat per...,local_per_series_optimal,Per-series local and zero-shot models are opti...,"n_series=4, scale_regime='micro', cross_series...",Global pooling pays off most when n_series >= ...
4,hierarchy_reconciliation,"5. Is hierarchical reconciliation coherent, no...",not_applicable_flat_panel,Flat panel (no hierarchy columns); output.floo...,hierarchy_id_cols=[],"Inspect f.hierarchy_value_df() for Root, Inter..."
5,horizon_stability,6. Does accuracy hold up across the whole fore...,horizon_specialist_mitigated,Late-horizon error grows 1.61x vs. early steps...,"late_early_ratio=1.61, crossover_step=None, di...",Inspect f.horizon_decay_df() for step-by-step ...
6,forecast_trust,7. How do I trust this forecast in production?,high_confidence,Evaluated on 1 rolling out-of-fold backtest wi...,"n_folds=1, champion_pass=3/4, residual_acf1=0....",Inspect f.explain_auto_decisions_df() for the ...



Head-to-Head Model Comparison (`f.compare_models()` on `dgp_04_true_vs_decoy_covs`):


,model_a,model_b,decision_metric,score_a,score_b,lift_pct,series_win_rate_pct,dm_pvalue,significant_at_10pct,verdict
0,ensemble_champion,ensemble_horizon_specialist,wape,0.0398,0.0443,10.26,50.0,None,False,ensemble_champion beats ensemble_horizon_speci...
1,ensemble_champion,ensemble_inverse_error,wape,0.0398,0.0443,10.26,50.0,None,False,ensemble_champion beats ensemble_inverse_error...
2,ensemble_champion,regression_lags@transform_log1p,wape,0.0398,0.0454,12.35,100.0,None,False,ensemble_champion beats regression_lags@transf...
3,ensemble_champion,regression_lags,wape,0.0398,0.0461,13.75,100.0,None,False,ensemble_champion beats regression_lags by +13...
4,ensemble_champion,ensemble_mean,wape,0.0398,0.0513,22.51,75.0,None,False,ensemble_champion beats ensemble_mean by +22.5...
5,ensemble_champion,lightgbm@transform_log1p,wape,0.0398,0.0583,31.80,100.0,None,False,ensemble_champion beats lightgbm@transform_log...
6,ensemble_champion,lightgbm,wape,0.0398,0.0583,31.81,100.0,None,False,ensemble_champion beats lightgbm by +31.81% (d...
7,ensemble_champion,theta@transform_log1p,wape,0.0398,0.0834,52.29,100.0,None,False,ensemble_champion beats theta@transform_log1p ...
8,ensemble_champion,theta,wape,0.0398,0.0846,52.95,100.0,None,False,ensemble_champion beats theta by +52.95% (dire...
9,ensemble_champion,naive_seasonal,wape,0.0398,0.1078,63.08,100.0,None,False,ensemble_champion beats naive_seasonal by +63....



Hierarchy Reconciliation Value Audit (`dgp_08_hierarchical_nonneg`):


,hierarchy_level,n_nodes,reconciliation_method,decision_metric,unreconciled_score,reconciled_score,reconciliation_lift_pct,max_coherence_residual,negative_leaf_count
0,Level 0: Root (Total),1,mint_nonneg,wape,0.0371,0.0368,0.80,0.0,0
1,Level 1: Intermediate,5,mint_nonneg,wape,0.0496,0.0489,1.31,0.0,0
2,Level 2: Leaf Series,4,mint_nonneg,wape,0.0527,0.0519,1.40,0.0,0
3,All Hierarchy Nodes,10,mint_nonneg,wape,0.0496,0.0489,1.31,0.0,0


## Act 6 · Opt-In Evolutionary Search, Post-Run Super-Ensembling & Causal Leakage Guards (`Forecaster.evolve`)

What if you already have one or more completed runs—including **manual `RunConfig` runs where you never used `Forecaster.auto()` or `autofocus=True`**—and want to discover a custom regime/horizon-aware ensemble or domain feature transformation that beats every built-in model and stacker?

`Forecaster.evolve()` (`scale_forecasting.evolve`) provides an opt-in evolutionary harness with three key capabilities:

1. **Post-Run (`f.evolve()`) & Multi-Run (`from_runs` / `from_run_ids`) Super-Ensembling — Zero Base-Model Refitting:**
   - Call `f.evolve(target="ensemble")` on any completed `Forecaster` instance, or pool out-of-fold predictions across multiple runs via `Forecaster.evolve(from_runs=[f_a, f_b], target="ensemble")` (or `from_run_ids=["run-1", "run-2"]` against BigQuery).
   - The harness verifies dataset/horizon/frequency compatibility (`verify_oof_pool_compatibility`), deduplicates identical `make_model_spec_hash` fits (`backtest` and `hpo` strictly hashed; `ensemble` and `hierarchy` excluded), disambiguates colliding model names with distinct HPO/feature specs as `<model>@<run_short_id>`, benchmarks against the **reigning `#1` model or built-in `ensemble_*` strategy**, and materializes calibrated future horizon forecasts (`res.predictions_df`).
2. **The 3-Gate Anti-Overfitting Ratchet & `+1e6` Future-Leakage Sentinel:**
   - **Pre-Gate AST Sandbox & `+1e6` Causality Probe (`verify_no_future_leakage`):** Blocks forbidden imports/syscalls and perturbs future targets $y_{t > t_0}$ by `+1,000,000`; if any historical feature at $t \le t_0$ shifts by $> 10^{-12}$, the candidate is rejected before scoring.
   - **Gate 1 (Inner CV Fold Gain) + Gate 2 (Diebold–Mariano $p$-value & Worst-Decile Tail Floor) + Gate 3 (Unseen Held-Out Lockdown Fold):** Splits backtest folds temporally into inner evolution folds and a held-out final lockdown fold. A candidate is promoted (`res.promoted == True`) *only* if all three gates pass; otherwise it ratchets safely back to the reigning champion (`res.promoted == False`).
3. **Configurable Cloud & Offline Backends (`backend` & `model_name`):**

| Backend | Endpoint & Licensing | Default `model_name` & Override Options |
| :--- | :--- | :--- |
| **`backend="alphaevolve"`** *(Default)* | `discoveryengine.googleapis.com/v1alpha` (`:acquirePrograms` $\leftrightarrow$ `:submitProgramsEvaluations`). Requires a Google Cloud organization with **Gemini Enterprise licensing + AlphaEvolve entitlement** (`enable_alphaevolve = true` in Terraform). | `model_name="gemini-3.8-flash"` default; set `model_name="gemini-4-argon"` for Pro-tier deep structural synthesis. |
| **`backend="vertex_gemini"`** *(Cloud Backup)* | `aiplatform.googleapis.com` (`generateContent` with an local island-model evolutionary buffer). Works in **any standard Google Cloud project** with Vertex AI enabled. | `model_name="gemini-3.8-flash"` default; supports `"gemini-4-argon"` or any custom Vertex AI Gemini model ID. |
| **`backend="mock"`** *(Offline / CI)* | Deterministic in-memory mutation generator with zero network or cloud calls. | Used below so this notebook runs 100% offline at `\$0.00` cost. |

In [6]:
import numpy as np
from scale_forecasting.evolve import (
    DEFAULT_GEMINI_MODEL,
    compile_candidate_symbol,
    validate_candidate_ast,
    verify_no_future_leakage,
)

print(f"Default evolutionary LLM model: {DEFAULT_GEMINI_MODEL!r} (override via model_name='gemini-4-argon')")

# 1. Post-Run Super-Ensembling on an existing Forecaster session (zero base-model refitting!)
f_completed = sessions["dgp_04_true_vs_decoy_covs"]
res_single = f_completed.evolve(
    target="ensemble",
    backend="mock",
    max_evaluations=4,
)
print(
    f"\n=== 1. Single-Run Post-Run Evolve (`f.evolve(target='ensemble')`) ===\n"
    f"Reigning #1 baseline score: {res_single.baseline_score:.4f} -> "
    f"Inner CV score: {res_single.inner_cv_score:.4f} (FVA: {res_single.fva_pct:+.2f}%) | "
    f"Lockdown: {res_single.lockdown_baseline_score:.4f} -> {res_single.lockdown_candidate_score:.4f} | "
    f"promoted={res_single.promoted}"
)
display(res_single.leaderboard_df)

# 2. Multi-Run Super-Ensembling across two distinct runs (`from_runs=[f_run1, f_run2]`)
# Run two separate model families on the same panel and pool their OOF predictions:
ds_pool = load_benchmark("dgp_02_multiplicative_log", n_series=4)
f_run1 = Forecaster.auto(
    ds_pool.panel_df,
    tier="first_light",
    where="local",
    autofocus=False,
    models=["naive_seasonal", "theta"],
)
f_run2 = Forecaster.auto(
    ds_pool.panel_df,
    tier="first_light",
    where="local",
    autofocus=False,
    models=["naive_seasonal", "lightgbm", "regression_lags"],
)

res_multi = Forecaster.evolve(
    from_runs=[f_run1, f_run2],
    target="ensemble",
    backend="mock",
    max_evaluations=4,
)
print(
    f"\n=== 2. Multi-Run Super-Ensemble Pool (`Forecaster.evolve(from_runs=[f_run1, f_run2])`) ===\n"
    f"Pooled models: {res_multi.diagnostics.get('models_pooled')} | "
    f"Reigning #1 champion: {res_multi.diagnostics.get('reigning_champion_model')!r}\n"
    f"Baseline score: {res_multi.baseline_score:.4f} -> "
    f"Inner CV score: {res_multi.inner_cv_score:.4f} (FVA: {res_multi.fva_pct:+.2f}%) | "
    f"Materialized predictions_df rows: {len(res_multi.predictions_df)}"
)
display(res_multi.predictions_df.head(6))

# 3. Verify the +1e6 Future-Leakage Sentinel catches look-ahead bugs immediately
causal_fn_code = """
def custom_features(df):
    out = df.copy()
    out["lag_1_safe"] = out.groupby("ts_id")["y"].shift(1).bfill()
    return out
"""

leaky_fn_code = """
def custom_features(df):
    out = df.copy()
    # BUG: negative shift peeks into future target values!
    out["future_leak"] = out.groupby("ts_id")["y"].shift(-1).ffill()
    return out
"""

assert validate_candidate_ast(causal_fn_code).valid
causal_fn = compile_candidate_symbol(causal_fn_code, "custom_features")
leaky_fn = compile_candidate_symbol(leaky_fn_code, "custom_features")

causal_ok = verify_no_future_leakage(causal_fn, ds_pool.panel_df, horizon=14)
leaky_ok = verify_no_future_leakage(leaky_fn, ds_pool.panel_df, horizon=14)
print(
    f"\n=== 3. Causal +1e6 Future-Leakage Probe (`verify_no_future_leakage`) ===\n"
    f"Causal lag-1 feature passed: {causal_ok} (expected True)\n"
    f"Leaky future-peek feature passed: {leaky_ok} (expected False — blocked before Gate 1!)"
)

Default evolutionary LLM model: 'gemini-3.8-flash' (override via model_name='gemini-4-argon')



=== 1. Single-Run Post-Run Evolve (`f.evolve(target='ensemble')`) ===
Reigning #1 baseline score: 0.0405 -> Inner CV score: 0.0339 (FVA: +16.14%) | Lockdown: 0.0384 -> 0.0372 | promoted=True


,candidate_id,generation,target,backend,ast_valid,causal_valid,inner_cv_score,baseline_score,inner_fva_pct,dm_pvalue,worst_decile_fva_pct,gate1_passed,gate2_passed,lockdown_baseline_score,lockdown_candidate_score,gate3_passed,promoted,status,rejection_reason,wall_seconds
0,cand_ensemble_g01,1,ensemble,mock,True,True,0.033931,0.04046,16.1374,0.002607,0.0868,True,True,0.038372,0.037161,True,True,promoted_champion,None,0.4091
1,cand_ensemble_g02,2,ensemble,mock,True,True,0.033931,0.04046,16.1374,0.002607,0.0868,True,True,0.038372,NaN,False,False,passed_inner_gates,None,0.1819
2,cand_ensemble_g03,3,ensemble,mock,True,True,0.033931,0.04046,16.1374,0.002607,0.0868,True,True,0.038372,NaN,False,False,passed_inner_gates,None,0.1791
3,cand_ensemble_g04,4,ensemble,mock,True,True,0.033931,0.04046,16.1374,0.002607,0.0868,True,True,0.038372,NaN,False,False,passed_inner_gates,None,0.1917



=== 2. Multi-Run Super-Ensemble Pool (`Forecaster.evolve(from_runs=[f_run1, f_run2])`) ===
Pooled models: ['ensemble_inverse_error@03077033', 'ensemble_inverse_error@15ae2bbf', 'ensemble_mean@03077033', 'ensemble_mean@15ae2bbf', 'lightgbm', 'naive_seasonal', 'regression_lags', 'theta'] | Reigning #1 champion: 'regression_lags'
Baseline score: 0.1334 -> Inner CV score: 0.1241 (FVA: +6.96%) | Materialized predictions_df rows: 56


,ts_id,forecast_date,yhat,yhat_lower,yhat_upper,model_type
0,series_001,2025-05-21,913.690003,791.397707,1035.982300,regression_lags
1,series_001,2025-05-22,827.879497,705.587200,950.171794,regression_lags
2,series_001,2025-05-23,655.995865,533.703568,778.288161,regression_lags
3,series_001,2025-05-24,570.076942,447.784646,692.369239,regression_lags
4,series_001,2025-05-25,492.273345,369.981048,614.565642,regression_lags
5,series_001,2025-05-26,499.696756,377.404460,621.989053,regression_lags



=== 3. Causal +1e6 Future-Leakage Probe (`verify_no_future_leakage`) ===
Causal lag-1 feature passed: True (expected True)
Leaky future-peek feature passed: False (expected False — blocked before Gate 1!)


<!-- sf-closing -->
---

## Where next

- **Read the full guides:** [Auto-forecaster & `autofocus` guide](https://statmike.github.io/scale-forecasting/auto_forecaster/) · [AlphaEvolve & Vertex Gemini evolutionary harness (`scale_forecasting.evolve`)](https://statmike.github.io/scale-forecasting/alphaevolve_runner/) · [Forecaster workbench](https://statmike.github.io/scale-forecasting/forecaster_workbench/) · [Models & ensembles reference](https://statmike.github.io/scale-forecasting/models_reference/)
- **Scale & validation proof:** [System validation ledger](https://statmike.github.io/scale-forecasting/validation/) · [Quota, sizing & 100k scale proof](https://statmike.github.io/scale-forecasting/quota_and_scale/)
- **Hands-on team lab:** [Workshop guide](https://statmike.github.io/scale-forecasting/workshop/)